<a href="https://colab.research.google.com/github/LRcyber/U-Net-VHP/blob/main/2026_05_04_REconstru3d_img_delta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Célula 1: INSTALAÇÃO DAS DEPENDÊNCIAS
!pip install -q trimesh
!pip install -q opencv-python-headless
!pip install -q scikit-image
!pip install -q open3d
!pip install -q plotly
!pip install -q matplotlib

print("✅ Dependências instaladas!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 107.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 129.3 MB/s eta 0:00:00
✅ Dependências instaladas!


In [2]:
# Célula 2: IMPORTAÇÕES E CONFIGURAÇÃO DO SPACING
import numpy as np
import cv2
import trimesh
from skimage import measure
import matplotlib.pyplot as plt
from google.colab import files
import os
import zipfile
import json
import re
from scipy import ndimage

print("=" * 80)
print("🏗️ RECONSTRUÇÃO 3D - VISIBLE HUMAN PROJECT")
print("=" * 80)

# CONFIGURAÇÃO DO SPACING (Visible Human Project)
SPACING_XY = 0.33  # mm por pixel (0.33 mm)
SPACING_Z = 1.0    # mm por fatia (1.0 mm)

print(f"\n📏 CONFIGURAÇÃO ESPACIAL:")
print(f"   Resolução XY: {SPACING_XY} mm/pixel")
print(f"   Espaçamento entre fatias: {SPACING_Z} mm")
print(f"   Fator de escala: 1 unidade = {SPACING_XY} mm")


🏗️ RECONSTRUÇÃO 3D - VISIBLE HUMAN PROJECT

📏 CONFIGURAÇÃO ESPACIAL:
   Resolução XY: 0.33 mm/pixel
   Espaçamento entre fatias: 1.0 mm
   Fator de escala: 1 unidade = 0.33 mm


In [3]:
# Célula 3: MONTAR GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
# Célula 4: LOCALIZAR AS IMAGENS RESULTANTES
output_dir = "/content/drive/My Drive/visible_human_delta_20260604_221354"

print(f"\n📁 Procurando imagens em: {output_dir}")

if not os.path.exists(output_dir):
    print("❌ Pasta não encontrada! Procurando alternativas...")
    for root, dirs, files in os.walk("/content/drive/My Drive"):
        if "visible_human_delta_" in root:
            output_dir = root
            print(f"✅ Pasta encontrada: {output_dir}")
            break
else:
    print("✅ Pasta encontrada!")



📁 Procurando imagens em: /content/drive/My Drive/visible_human_delta_20260604_221354
✅ Pasta encontrada!


In [5]:
# Célula 5: CARREGAR IMAGENS
print("\n📊 Carregando imagens resultantes...")

imagens_paths = []
for root, dirs, files in os.walk(output_dir):
    for file in files:
        if file == 'delta_resultado.png':
            imagens_paths.append(os.path.join(root, file))

def extrair_num_par(path):
    match = re.search(r'par_(\d+)', path)
    return int(match.group(1)) if match else 0

imagens_paths.sort(key=extrair_num_par)

print(f"✅ Encontradas {len(imagens_paths)} imagens")

if len(imagens_paths) == 0:
    print("❌ Nenhuma imagem encontrada!")
    raise SystemExit("Parando execução")

# Usar todas as imagens disponíveis
MAX_IMAGENS = len(imagens_paths)
print(f"📚 Usando todas as {MAX_IMAGENS} imagens")


📊 Carregando imagens resultantes...
✅ Encontradas 1876 imagens
📚 Usando todas as 1876 imagens


In [ ]:
# Célula 6: CRIAR VOLUME 3D OTIMIZADO (SEM CARREGAR TUDO NA RAM)
print("\n🏗️ Criando volume 3D com spacing real - VERSÃO OTIMIZADA")

# Carregar primeira imagem para dimensões
img_sample = cv2.imread(imagens_paths[0])
h, w = img_sample.shape[:2]
print(f"📐 Dimensões das imagens: {w} x {h} pixels")
print(f"📏 Dimensões físicas: {w * SPACING_XY:.1f} x {h * SPACING_XY:.1f} mm")
print(f"📏 Profundidade total: {len(imagens_paths) * SPACING_Z:.1f} mm")

# Estimar memória necessária (para alerta)
memoria_gb_estimada = (len(imagens_paths) * h * w * 1) / (1024**3)  # Apenas volume binário
print(f"⚠️ Memória estimada para volume completo: {memoria_gb_estimada:.1f} GB")

if memoria_gb_estimada > 8:
    print("❌ Isso ultrapassaria a RAM disponível!")
    print("✅ Usando abordagem de processamento em lote...")

# Abordagem 1: Processamento em lote com salvamento em disco
print("\n💾 Usando processamento em disco (não carrega tudo na RAM)")

# Criar arquivo temporário no disco para o volume
import tempfile
volume_path = tempfile.mktemp(suffix='.npy')
colors_path = tempfile.mktemp(suffix='.npy')

# Processar em lotes
BATCH_SIZE = 50  # Processar 50 imagens por vez
total_batches = (len(imagens_paths) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"📦 Processando em {total_batches} lotes de {BATCH_SIZE} imagens")

# Primeiro, determinar dimensões totais
volume_shape = (len(imagens_paths), h, w)
print(f"📊 Shape do volume: {volume_shape}")

# Criar arrays no disco (memory-mapped)
volume_mmap = np.memmap(volume_path, dtype='uint8', mode='w+', shape=volume_shape)
colors_mmap = np.memmap(colors_path, dtype='uint8', mode='w+', shape=(len(imagens_paths), h, w, 3))

print("🔄 Processando camadas em lotes...")
for batch_idx in range(total_batches):
    start_idx = batch_idx * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, len(imagens_paths))

    print(f"   Lote {batch_idx + 1}/{total_batches} (imagens {start_idx} a {end_idx})")

    for idx in range(start_idx, end_idx):
        img_path = imagens_paths[idx]
        img = cv2.imread(img_path)

        if img is None:
            continue

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Detectar pixels não-brancos
        is_not_white = ~(np.all(img_rgb == [255, 255, 255], axis=2))

        # Salvar no memory-mapped array
        volume_mmap[idx] = is_not_white.astype(np.uint8) * 255
        colors_mmap[idx] = img_rgb

        # Liberar memória da imagem
        del img, img_rgb, is_not_white

    # Forçar flush para disco
    volume_mmap.flush()
    colors_mmap.flush()

    # Mostrar uso de memória
    import psutil
    memoria_used = psutil.virtual_memory().percent
    print(f"      💾 Memória usada: {memoria_used}%")
    print(f"      📦 Processadas {end_idx}/{len(imagens_paths)} camadas")

print(f"✅ Volume 3D criado no disco (não na RAM)")


🏗️ Criando volume 3D com spacing real - VERSÃO OTIMIZADA
📐 Dimensões das imagens: 2048 x 1216 pixels
📏 Dimensões físicas: 675.8 x 401.3 mm
📏 Profundidade total: 1876.0 mm
⚠️ Memória estimada para volume completo: 4.4 GB

💾 Usando processamento em disco (não carrega tudo na RAM)
📦 Processando em 38 lotes de 50 imagens
📊 Shape do volume: (1876, 1216, 2048)
🔄 Processando camadas em lotes...
   Lote 1/38 (imagens 0 a 50)
      💾 Memória usada: 11.0%
      📦 Processadas 50/1876 camadas
   Lote 2/38 (imagens 50 a 100)
      💾 Memória usada: 10.9%
      📦 Processadas 100/1876 camadas
   Lote 3/38 (imagens 100 a 150)
      💾 Memória usada: 11.1%
      📦 Processadas 150/1876 camadas
   Lote 4/38 (imagens 150 a 200)
      💾 Memória usada: 11.1%
      📦 Processadas 200/1876 camadas
   Lote 5/38 (imagens 200 a 250)
      💾 Memória usada: 10.8%
      📦 Processadas 250/1876 camadas
   Lote 6/38 (imagens 250 a 300)
      💾 Memória usada: 10.9%
      📦 Processadas 300/1876 camadas
   Lote 7/38 (image